In [1]:
import re
import yaml
import json
import os

class LiteralString(str):
    pass
def literal_representer(dumper,value):
    return dumper.represent_scalar('tag:yaml.org,2002:str', value, style='|')

# Register custom representer
yaml.add_representer(LiteralString, literal_representer)

class Helper:
    @staticmethod
    def load_file(filepath: str) -> str:
        with open(filepath, "r", encoding="utf-8") as file:
            return file.read()
    @staticmethod
    def load_yaml(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)
    @staticmethod
    def load_json(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    @staticmethod
    def save_yaml(newconfig,filepath: str):
        with open(filepath,"w",encoding="utf-8") as file:
            return yaml.dump(newconfig,file,sort_keys=False)
    @staticmethod
    def fop(num: float) -> float:
        return float(f"{num:.1f}")
    @staticmethod
    def prettyjson(txt:str) -> str:
        return str(json.dumps(txt,indent=4, ensure_ascii=False))
    @staticmethod
    def to_literal(value):
        if isinstance(value,str) and "\n" in value:
            return LiteralString(value)
        return value
    @staticmethod
    def deep_literal_transform(data):
        if isinstance(data, dict):
            return {k: Helper.deep_literal_transform(v) for k,v in data.items()}
        if isinstance(data, list):
            return [Helper.deep_literal_transform(i) for i in data]
        return Helper.to_literal(data)

In [2]:
import asyncio
from pydantic import BaseModel, Field, ValidationError
from typing import Dict
from google.genai import types

class CriterionScore(BaseModel):
    score: int = Field(ge=0, le=5)
    feedback: str

class SectionEvaluation(BaseModel):
    section: str
    scores: Dict[str, CriterionScore]
    session_feedback: str

class LlmCaller(Helper):
    def __init__(self):
        self.client    = genai.Client(api_key="AIzaSyB...2V1wFsGycVM")
        self.model_cfg = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")
        self.model     = self.model_cfg["model"]["generation_model"]
        self.usd2bath  = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['currency']['USD_to_THB']
        self.log_digit = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['logging']['logging_round_digit']
    def extract_json(text: str) -> dict:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("NO_JSON_OBJECT_FOUND")
        return json.loads(match.group())
    def _parse(self, resp):
        text = resp.text.strip()
        text = re.sub(r"^```json|```$", "", text).strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError as e:
            pass        
        try:
            return self.extract_json(text)
        except Exception as e:
            raise ValueError(f"INVALID_JSON::{text}") from e
    def _call_raw(self, prompt: str):
        resp = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
    #         config=types.GenerateContentConfig(
    #             # thinking_config=types.ThinkingConfig(thinking_budget=1024)
    #             # Turn off thinking:
    #             thinking_config=types.ThinkingConfig(thinking_budget=0)
    #             # Turn on dynamic thinking:
    #             # thinking_config=types.ThinkingConfig(thinking_budget=-1)
    # )
)
        parsed = self._parse(resp)
        return parsed, resp
    def _validate(self, raw_output:dict)->SectionEvaluation:
        return SectionEvaluation.model_validate(raw_output)
    def _repair_prompt(self, error_msg: str) -> str:
        return f"""
                Your previous response was INVALID.
                Validation error:
                {error_msg}
                STRICT RULES:
                - Return JSON only
                - No markdown
                - No explanation
                - Follow schema exactly
                - Section name must start with a capital letters (e.g. "Education")
                Expected format:
                {{
                    "section": "<Section_name>",
                    "scores": {{
                        "<criterion>": {{ 
                            "score": 0-5, 
                            "feedback": "string" 
                        }}
                    }},
                    "session_feedback":"string"
                }}
        """
    def call(self,prompt:str, max_retry:int = 3):
        last_error = None
        repair_prompt = "\n"
        for attemp in range(max_retry):
            final_prompt = repair_prompt + prompt
            # print(f"final_prompt attemp : {attemp} -> \n {final_prompt}")
            # print(f"Output -> \n{output}")
            try:
                output, raw = self._call_raw(final_prompt)
                validated   = self._validate(output)
                # print('Status : 1')
                return validated.model_dump(),raw
            except (ValidationError, ValueError) as e:
                last_error = str(e)
                repair_prompt = self._repair_prompt(last_error)
                # print('Status : 0')
                print("Output error recall again ...")
            finally:
                print('='*100)
        return {
            "section":"UNKNOW",
            "scores":{}
        },raw
    async def call_async(self, prompt: str):
        return await asyncio.to_thread(self.call, prompt)

In [97]:
from google import genai
import json
import os
import yaml


class BasePromptBuilder(Helper):
    '''
    PromptBuilder v3 : PromptBuilder + Session,Global feedback + PromptSplit
    '''
    base_dir = r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts"         # Prompts .yaml config file folder path
    def __init__(self, section, criteria, targetrole, cvresume, include_fewshot: bool = True, output_lang = "en"):
        self.section         = section
        self.criteria        = criteria[::-1]
        self.targetrole      = targetrole
        self.cvresume        = cvresume
        self.include_fewshot = include_fewshot
        self.output_lang     = output_lang
        if self.targetrole == "NA" and 'RoleRelevance' in self.criteria:
            self.criteria.remove("RoleRelevance")
        if self.targetrole in ["", "NA", None]:
            self.targetrole = "__NO_ROLE__"

        self.global_config   = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts\base.yaml")
        self.section_config  = self.load_yaml(f"{self.base_dir}/{self.section.lower()}.yaml")
        self.number_of_words = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")["output"]["number_of_words"]
        self.criteria_cfg    = self.section_config['criteria']
        self.output_gd_cfg   = self.section_config['output_guidelines']
    def _build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            },
            "session_feedback":""
        }
    
    def _build_criteria_block(self) -> str:
        blocks = []
        for crit in self.criteria:
            block = f"- {crit}\n"
            for i in [5,3,1]:
                block += f"  score {i} :\n"
                x = f"score{i}"
                block += "\n".join(
                    "    " + line
                    for line in self.criteria_cfg[crit][x].splitlines()
                ) + "\n"
            blocks.append(block)
        return "".join(blocks)

    def _build_output_guideline_block(self) -> str:
        blocks = []
        for crit in self.criteria:
            block = f"- {crit}\n"
            for i in [5,3,1]:
                block += f"  score {i} :\n"
                x = f"score{i}"
                block += "\n".join(
                    "    " + line
                    for line in self.output_gd_cfg[crit][x].splitlines()
                ) + "\n"
            blocks.append(block)
        return "\n".join(b.strip() for b in blocks) + "\n"


    def build(self):
        config_role       = self.global_config['role']['role1']
        config_task       = self.section_config['task']['task1']
        config_lang       = self.global_config['Language_output_style'][self.output_lang]
        config_expected   = self.section_config['expected_content'][self.section]
        criteria_block    = self._build_criteria_block()
        config_example    = self._build_output_guideline_block()
        config_scale      = self.global_config['scale']['score1']
        config_feedback   = self.global_config['feedback']['globalfeedback']

        prompt_role       = f"Role :\n{config_role}\n\n"
        prompt_task       = f"Task :\n{config_task}\n"
        prompt_lang       = f"Output language instruction :\n{config_lang}\n"
        prompt_expected   = f"Expected :\n{config_expected}\n"
        prompt_criteria   = f"Criteria :\n{criteria_block}\n"
        prompt_scale      = f"Scale :\n{config_scale}\n"
        prompt_feedback   = f"Session feedback :\n{config_feedback}\n\n"
        prompt_Op_example = f"Output guideline :\n{config_example}\n\n"
        prompt_Op_format  = f"Output format :\n{json.dumps(self._build_response_template(), indent=2)}\n\n"
        prompt_cvresume   = f"CV/Resume :\n{self.cvresume}\n"

        prompt = (
            prompt_role + prompt_task + prompt_lang
            + prompt_expected + prompt_criteria + prompt_scale + prompt_feedback 
            + prompt_Op_example + prompt_Op_format + prompt_cvresume 
        )

        prompt = prompt.replace("<section_name>", self.section)
        prompt = prompt.replace("<targetrole>", self.targetrole)
        prompt = prompt.replace("<number_of_words>", str(self.number_of_words))

        return prompt

In [98]:
p4 = BasePromptBuilder( 
        section     = "Skills", 
        criteria    = ["Completeness","Length","RoleRelevance"],
    targetrole  = "NA",
    cvresume    = "resume_json",
    output_lang = "en" 
)
prompt2 = p4.build()
print(prompt2)

Role :
You are the expert HR evaluator

Task :
Evaluate the Skills section of the resume using the scoring criteria
and assess how well the listed skills support the __NO_ROLE__ role.
Consider completeness, relevance, and appropriateness of skill quantity.

Output language instruction :
- All feedback text MUST be written in English.
- Scores MUST remain numeric.
- JSON keys MUST remain in English exactly as defined
- Use the example outputs only as style and format references. Do NOT reuse their content.
- Each feedback MUST be a single sentence with a maximum of 20 words.

Expected :
- Technical skills (e.g., programming languages, tools, frameworks)
- Domain or functional skills relevant to the target role
- Soft skills where appropriate
- Clear grouping or categorization of skills
- Appropriate number of skills (neither too few nor excessive)

Criteria :
- Length
  score 5 :
    - Number of skills is appropriate (typically 10-20)
    - Easy to scan without being overwhelming
    - 